## Part B - From-Scratch Implementation

### 1. Import libraries and load the dataset

In [58]:
import pandas as pd
import numpy as np
import time

# Load the dataset
df = pd.read_csv("data/garments_worker_productivity.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


### 2. Prepare the dataset

In [59]:
# Make a copy so the original dataset is not changed
df_model = df.copy()

# Remove the date column
df_model = df_model.drop(columns=["date"])

# Create classification target
df_model["MeetsTarget"] = (
    df_model["actual_productivity"] >= df_model["targeted_productivity"]
).astype(int)

print("MeetsTarget distribution:")
print(df_model["MeetsTarget"].value_counts())

print("\nMissing values:")
print(df_model.isnull().sum())

MeetsTarget distribution:
MeetsTarget
1    875
0    322
Name: count, dtype: int64

Missing values:
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
MeetsTarget                0
dtype: int64


### 3. Separate features and targets

In [60]:
# Features
X = df_model.drop(columns=["actual_productivity", "MeetsTarget"])

# Regression target
y_reg = df_model["actual_productivity"].values

# Classification target
y_cls = df_model["MeetsTarget"].values

print("Feature shape:", X.shape)
print("Regression target shape:", y_reg.shape)
print("Classification target shape:", y_cls.shape)

Feature shape: (1197, 13)
Regression target shape: (1197,)
Classification target shape: (1197,)


### 4. Handle missing values manually

In [61]:
numeric_columns = X.select_dtypes(
    include=["int64", "float64", "bool"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical columns:")
print(numeric_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Categorical columns:
['quarter', 'department', 'day']


C:\Users\George Mathew\AppData\Local\Temp\ipykernel_2556\3008114648.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


In [62]:
for column in numeric_columns:
    X[column] = X[column].fillna(X[column].median())

# Categorical columns → most frequent value
for column in categorical_columns:
    X[column] = X[column].fillna(X[column].mode()[0])

print("Remaining missing values:")
print(X.isnull().sum().sum())

Remaining missing values:


0


### 5. Encode categorical variables manually

In [63]:
X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=False,
    dtype=float
)

print("Shape after encoding:", X_encoded.shape)

X_encoded.head()

Shape after encoding: (1197, 24)


,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,...,quarter_Quarter5,department_finishing,department_finishing,department_sweing,day_Monday,day_Saturday,day_Sunday,day_Thursday,day_Tuesday,day_Wednesday
0,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,1,0.75,3.94,1039.0,960,0,0.0,0,0,8.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


### 6. Convert everything to NumPy

In [64]:
X_array = X_encoded.to_numpy(dtype=float)

print("X shape:", X_array.shape)
print("X data type:", X_array.dtype)

X shape: (1197, 24)
X data type: float64


In [65]:
# ============================================
# USE THE SAME TRAIN-TEST SPLIT AS SKLEARN
# ============================================

# Load the train-test indices created in
# Lab5_Sklearn.ipynb

train_indices = np.load("train_indices.npy")
test_indices = np.load("test_indices.npy")


# Create training and testing data
X_train = X_array[train_indices]
X_test = X_array[test_indices]

y_reg_train = y_reg[train_indices]
y_reg_test = y_reg[test_indices]

y_cls_train = y_cls[train_indices]
y_cls_test = y_cls[test_indices]


# ============================================
# CHECK THE SPLIT
# ============================================

print("Training samples:", len(train_indices))
print("Testing samples:", len(test_indices))

print("\nTraining data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training samples: 957
Testing samples: 240

Training data shape: (957, 24)
Testing data shape: (240, 24)


In [66]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

std[std == 0] = 1

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

print("Training data mean:")
print(X_train_scaled.mean(axis=0)[:10])

print("\nTraining data standard deviation:")
print(X_train_scaled.std(axis=0)[:10])

Training data mean:
[ 5.58011781e-17  1.28925954e-14 -2.30420582e-15 -9.51070756e-17
  3.01627990e-18  1.20477180e-16 -1.49885909e-16 -8.61959987e-17
  2.21232530e-16 -1.48493780e-16]

Training data standard deviation:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### 9. Add an intercept column

In [67]:
X_train_final = np.column_stack(
    [np.ones(X_train_scaled.shape[0]), X_train_scaled]
)

X_test_final = np.column_stack(
    [np.ones(X_test_scaled.shape[0]), X_test_scaled]
)

print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)

Final training shape: (957, 25)
Final testing shape: (240, 25)


### Manual Linear Regression

### 1. Train Linear Regression manually

In [68]:
# ============================================
# MANUAL LINEAR REGRESSION
# ============================================

# Closed-form solution:
# beta = (X^T X)^(-1) X^T y

start_time = time.perf_counter()

# Calculate the regression coefficients
XTX = X_train_final.T @ X_train_final
XTy = X_train_final.T @ y_reg_train

beta = np.linalg.pinv(XTX) @ XTy

linear_training_time_manual = time.perf_counter() - start_time

print("Linear Regression training completed.")
print("Number of coefficients:", len(beta))
print("Training time:", linear_training_time_manual, "seconds")

Linear Regression training completed.
Number of coefficients: 25
Training time: 0.004210999992210418 seconds


### 2. Make predictions

In [69]:
# Make predictions on the test data

start_time = time.perf_counter()

y_reg_pred_manual = X_test_final @ beta

linear_prediction_time_manual = time.perf_counter() - start_time

print("Prediction completed.")
print("Prediction time:", linear_prediction_time_manual, "seconds")
print("\nFirst 10 predictions:")
print(y_reg_pred_manual[:10])

Prediction completed.
Prediction time: 0.00024850000045262277 seconds

First 10 predictions:
[0.83653561 0.7075372  0.78752045 0.63881458 0.73453934 0.75485423
 0.67627679 0.79155521 0.74639686 0.78273093]


### 3. Calculate MAE manually

In [70]:
# Mean Absolute Error (MAE)

mae_manual = np.mean(
    np.abs(y_reg_test - y_reg_pred_manual)
)

print("Manual MAE:", mae_manual)

Manual MAE: 0.10727843302917202


### 4. Calculate RMSE manually

In [71]:
# Root Mean Squared Error (RMSE)

rmse_manual = np.sqrt(
    np.mean((y_reg_test - y_reg_pred_manual) ** 2)
)

print("Manual RMSE:", rmse_manual)

Manual RMSE: 0.1437402979105884


### 5. Calculate R² manually

In [72]:
# R-squared (R2)

ss_res = np.sum(
    (y_reg_test - y_reg_pred_manual) ** 2
)

ss_tot = np.sum(
    (y_reg_test - np.mean(y_reg_test)) ** 2
)

r2_manual = 1 - (ss_res / ss_tot)

print("Manual R2:", r2_manual)

Manual R2: 0.2885214314434412


### 6. Display all Linear Regression results

In [73]:
print("======================================")
print("MANUAL LINEAR REGRESSION RESULTS")
print("======================================")

print("MAE:", mae_manual)
print("RMSE:", rmse_manual)
print("R2:", r2_manual)
print("Training time:", linear_training_time_manual, "seconds")
print("Prediction time:", linear_prediction_time_manual, "seconds")

MANUAL LINEAR REGRESSION RESULTS
MAE: 0.10727843302917202
RMSE: 0.1437402979105884
R2: 0.2885214314434412
Training time: 0.004210999992210418 seconds
Prediction time: 0.00024850000045262277 seconds


### Manual Logistic Regression

### 1. Sigmoid function

In [84]:
# ============================================
# SIGMOID FUNCTION
# ============================================

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

### 2. Train Logistic Regression using Gradient Descent

In [85]:
# Logistic Regression using Gradient Descent

start_time = time.perf_counter()

# Number of features
n_features = X_train_final.shape[1]

# Start all coefficients at zero
weights = np.zeros(n_features)

# Learning rate
learning_rate = 0.01

# Number of iterations
iterations = 5000

# Number of training samples
n = X_train_final.shape[0]

for i in range(iterations):

    # Calculate linear combination
    z = X_train_final @ weights

    # Convert to probabilities
    probabilities = sigmoid(z)

    # Calculate gradient
    gradient = (X_train_final.T @ (probabilities - y_cls_train)) / n

    # Update weights
    weights = weights - learning_rate * gradient

logistic_training_time_manual = time.perf_counter() - start_time

print("Logistic Regression training completed.")
print("Training time:", logistic_training_time_manual, "seconds")

Logistic Regression training completed.
Training time: 0.7180995999951847 seconds


### 3. Predict probabilities

In [86]:
# Predict probabilities for the test data

start_time = time.perf_counter()

z_test = X_test_final @ weights

probabilities_test = sigmoid(z_test)

logistic_prediction_time_manual = time.perf_counter() - start_time

print("Prediction completed.")
print("Prediction time:", logistic_prediction_time_manual, "seconds")

print("\nFirst 10 predicted probabilities:")
print(probabilities_test[:10])

Prediction completed.
Prediction time: 0.003916699992259964 seconds

First 10 predicted probabilities:
[0.59920073 0.83892021 0.81066069 0.3901345  0.89863185 0.82101793
 0.54746608 0.66487362 0.5330214  0.42880145]


### 4. Convert probabilities into 0 or 1

In [87]:
# Convert probabilities into class predictions

threshold = 0.5

y_cls_pred_manual = (
    probabilities_test >= threshold
).astype(int)

print("First 20 predictions:")
print(y_cls_pred_manual[:20])

First 20 predictions:
[1 1 1 0 1 1 1 1 1 0 0 1 1 1 0 1 1 1 1 1]


### 5. Calculate Accuracy manually

In [89]:
# Accuracy

accuracy_manual = np.mean(
    y_cls_pred_manual == y_cls_test
)

print("Manual Accuracy:", accuracy_manual)

Manual Accuracy: 0.7166666666666667


### 6. Calculate Precision manually

In [90]:
# Calculate confusion matrix components

true_positive = np.sum(
    (y_cls_pred_manual == 1) & (y_cls_test == 1)
)

false_positive = np.sum(
    (y_cls_pred_manual == 1) & (y_cls_test == 0)
)

false_negative = np.sum(
    (y_cls_pred_manual == 0) & (y_cls_test == 1)
)

true_negative = np.sum(
    (y_cls_pred_manual == 0) & (y_cls_test == 0)
)

print("True Positive:", true_positive)
print("False Positive:", false_positive)
print("False Negative:", false_negative)
print("True Negative:", true_negative)

True Positive: 157
False Positive: 50
False Negative: 18
True Negative: 15


In [91]:
# Precision

if (true_positive + false_positive) == 0:
    precision_manual = 0
else:
    precision_manual = (
        true_positive /
        (true_positive + false_positive)
    )

print("Manual Precision:", precision_manual)

Manual Precision: 0.7584541062801933


### 7. Calculate Recall manually

In [92]:
# Recall

if (true_positive + false_negative) == 0:
    recall_manual = 0
else:
    recall_manual = (
        true_positive /
        (true_positive + false_negative)
    )

print("Manual Recall:", recall_manual)

Manual Recall: 0.8971428571428571


### 8. Calculate F1-score manually

In [93]:
# F1-score

if (precision_manual + recall_manual) == 0:
    f1_manual = 0
else:
    f1_manual = (
        2 * precision_manual * recall_manual
        / (precision_manual + recall_manual)
    )

print("Manual F1-score:", f1_manual)

Manual F1-score: 0.8219895287958117


### 9. Final Logistic Regression results

In [94]:
print("======================================")
print("MANUAL LOGISTIC REGRESSION RESULTS")
print("======================================")

print("Accuracy:", accuracy_manual)
print("Precision:", precision_manual)
print("Recall:", recall_manual)
print("F1-score:", f1_manual)
print("Training time:", logistic_training_time_manual, "seconds")
print("Prediction time:", logistic_prediction_time_manual, "seconds")

MANUAL LOGISTIC REGRESSION RESULTS
Accuracy: 0.7166666666666667
Precision: 0.7584541062801933
Recall: 0.8971428571428571
F1-score: 0.8219895287958117
Training time: 0.7180995999951847 seconds
Prediction time: 0.003916699992259964 seconds


In [95]:
# ============================================
# SAVE FROM-SCRATCH RESULTS
# ============================================

manual_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Logistic Regression"
    ],
    
    "MAE": [
        mae_manual,
        np.nan
    ],
    
    "RMSE": [
        rmse_manual,
        np.nan
    ],
    
    "R2": [
        r2_manual,
        np.nan
    ],
    
    "Accuracy": [
        np.nan,
        accuracy_manual
    ],
    
    "Precision": [
        np.nan,
        precision_manual
    ],
    
    "Recall": [
        np.nan,
        recall_manual
    ],
    
    "F1": [
        np.nan,
        f1_manual
    ],
    
    "Training Time (s)": [
        linear_training_time_manual,
        logistic_training_time_manual
    ],
    
    "Prediction Time (s)": [
        linear_prediction_time_manual,
        logistic_prediction_time_manual
    ]
})

print("From-Scratch Results:")
display(manual_results)

# Save results
manual_results.to_csv(
    "manual_results.csv",
    index=False
)

print("\nResults saved to manual_results.csv")

From-Scratch Results:


,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Linear Regression,0.107278,0.14374,0.288521,NaN,NaN,NaN,NaN,0.004211,0.000249
1,Logistic Regression,NaN,NaN,NaN,0.716667,0.758454,0.897143,0.82199,0.718100,0.003917



Results saved to manual_results.csv


In [96]:
# ============================================
# CREATE AND SAVE FROM-SCRATCH RESULTS
# ============================================

manual_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Logistic Regression"
    ],

    "MAE": [
        mae_manual,
        np.nan
    ],

    "RMSE": [
        rmse_manual,
        np.nan
    ],

    "R2": [
        r2_manual,
        np.nan
    ],

    "Accuracy": [
        np.nan,
        accuracy_manual
    ],

    "Precision": [
        np.nan,
        precision_manual
    ],

    "Recall": [
        np.nan,
        recall_manual
    ],

    "F1": [
        np.nan,
        f1_manual
    ],

    "Training Time (s)": [
        linear_training_time_manual,
        logistic_training_time_manual
    ],

    "Prediction Time (s)": [
        linear_prediction_time_manual,
        logistic_prediction_time_manual
    ]
})

print("From-Scratch Results:")
display(manual_results)

# Save results
manual_results.to_csv(
    "manual_results.csv",
    index=False
)

print("\nResults saved to manual_results.csv")

From-Scratch Results:


,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Linear Regression,0.107278,0.14374,0.288521,NaN,NaN,NaN,NaN,0.004211,0.000249
1,Logistic Regression,NaN,NaN,NaN,0.716667,0.758454,0.897143,0.82199,0.718100,0.003917



Results saved to manual_results.csv
